# Gradient Descent Optimizers from Scratch

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/gradient-descent-optimizers)

We implement SGD, Momentum, Adagrad, RMSprop, and Adam from their update rules, race them on an ill-conditioned 'ravine', show Adagrad stalling, and verify Adam's bias correction.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## 1 — An ill-conditioned test function

$f(x,y) = 0.5(a x^2 + b y^2)$ with $a \gg b$ is a ravine: steep in x, flat in y. This is exactly where plain SGD struggles.

In [ ]:
a, b = 20.0, 1.0
f    = lambda p: 0.5*(a*p[0]**2 + b*p[1]**2)
grad = lambda p: np.array([a*p[0], b*p[1]])
start = np.array([-9.0, -9.0])

## 2 — The optimizers, each from its update rule

In [ ]:
def optimize(kind, eta, steps=60, **kw):
    p = start.copy()
    path = [p.copy()]
    m = np.zeros(2); v = np.zeros(2); G = np.zeros(2)
    eps = 1e-8
    for t in range(1, steps+1):
        g = grad(p)
        if kind == 'sgd':
            p = p - eta*g
        elif kind == 'momentum':
            m = kw.get('mu',0.9)*m + g
            p = p - eta*m
        elif kind == 'adagrad':
            G += g**2
            p = p - eta*g/(np.sqrt(G)+eps)
        elif kind == 'rmsprop':
            v = 0.9*v + 0.1*g**2
            p = p - eta*g/(np.sqrt(v)+eps)
        elif kind == 'adam':
            b1, b2 = 0.9, 0.999
            m = b1*m + (1-b1)*g
            v = b2*v + (1-b2)*g**2
            mhat = m/(1-b1**t); vhat = v/(1-b2**t)
            p = p - eta*mhat/(np.sqrt(vhat)+eps)
        path.append(p.copy())
    return np.array(path)

runs = {
    'SGD':      optimize('sgd', 0.04),
    'Momentum': optimize('momentum', 0.02),
    'RMSprop':  optimize('rmsprop', 0.3),
    'Adam':     optimize('adam', 0.6),
}

In [ ]:
# Contour + trajectories
xs = np.linspace(-10, 10, 200); ys = np.linspace(-10, 10, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5*(a*XX**2 + b*YY**2)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].contour(XX, YY, ZZ, levels=30, cmap='magma', alpha=0.5)
for name, path in runs.items():
    ax[0].plot(path[:,0], path[:,1], 'o-', ms=2, label=name)
ax[0].plot(0,0,'w*', ms=15); ax[0].set_title('Trajectories on a ravine'); ax[0].legend()
for name, path in runs.items():
    ax[1].semilogy([f(p) for p in path], label=name)
ax[1].set_xlabel('step'); ax[1].set_ylabel('loss (log)'); ax[1].set_title('Convergence'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3 — Adagrad stalls

Its accumulator only grows, so the effective rate decays to zero and progress halts.

In [ ]:
ada = optimize('adagrad', 0.5, steps=200)
rms = optimize('rmsprop', 0.5, steps=200)
plt.figure(figsize=(8,4))
plt.semilogy([f(p) for p in ada], label='Adagrad (stalls)', color='#f87171')
plt.semilogy([f(p) for p in rms], label='RMSprop (keeps going)', color='#34d399')
plt.xlabel('step'); plt.ylabel('loss (log)'); plt.legend()
plt.title('Adagrad vs RMSprop: the decaying-rate problem'); plt.tight_layout(); plt.show()
print(f"Adagrad final loss: {f(ada[-1]):.4e}   RMSprop final loss: {f(rms[-1]):.4e}")

## 4 — Why Adam needs bias correction

Without it the first steps are mis-scaled because m, v start at zero.

In [ ]:
b2 = 0.999; g = 1.0
for t in [1, 2, 10, 100, 1000]:
    v_raw = (1-b2)*g**2 * sum(b2**k for k in range(t))   # accumulated EMA of g^2=1
    correction = 1 - b2**t
    print(f"t={t:>4}: raw v={v_raw:.4f}  correction (1-b2^t)={correction:.4f}  corrected={v_raw/correction:.4f}")
print("\nRaw v is far too small early (biased toward 0); dividing by (1-b2^t) fixes the scale.")

## ✏️ Your turn

**Task A — AdamW:** Add decoupled weight decay to the Adam branch: $p \leftarrow p - \eta(\hat m/(\sqrt{\hat v}+\epsilon) + \lambda p)$. On a problem with a known nonzero optimum offset, show AdamW shrinks weights toward 0 differently than adding $\lambda p$ to the gradient (L2).

**Task B — Nesterov momentum:** Implement NAG by evaluating the gradient at the look-ahead point $p - \eta\mu m$ instead of at $p$, and compare its trajectory to plain momentum on the ravine.

In [ ]:
def optimize_adamw(eta, lam, steps=60):
    p = start.copy(); path=[p.copy()]
    m = np.zeros(2); v = np.zeros(2); b1,b2,eps = 0.9,0.999,1e-8
    for t in range(1, steps+1):
        g = grad(p)
        # TODO(you): Adam moments + bias correction, then decoupled weight decay
        path.append(p.copy())
    return np.array(path)

path = optimize_adamw(0.6, 0.01)
if len(path) > 1 and not np.allclose(path[-1], start):
    print(f"AdamW final loss: {f(path[-1]):.4e}")

<details><summary>Solution — Task A</summary>

```python
def optimize_adamw(eta, lam, steps=60):
    p = start.copy(); path=[p.copy()]
    m = np.zeros(2); v = np.zeros(2); b1,b2,eps = 0.9,0.999,1e-8
    for t in range(1, steps+1):
        g = grad(p)
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        mhat = m/(1-b1**t); vhat = v/(1-b2**t)
        p = p - eta*(mhat/(np.sqrt(vhat)+eps) + lam*p)   # decoupled decay
        path.append(p.copy())
    return np.array(path)
```
</details>